# Qwen3 LID Experiments on Colab

This notebook runs the local experiment suite on a Colab A100-40G runtime. Corrected experiment names are used: Experiment B is the prompt-structure no-thinking control, and Experiment C is the canonical think vs no-think comparison.

In [ ]:
# Setup project path and dependencies.
# If the repo is not already present on the Colab machine, set REPO_URL and run this cell.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # Example: "https://github.com/<user>/<repo>.git"
PROJECT_DIR = Path("/content/NLP_Final")

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL or upload/copy the project to /content/NLP_Final before continuing.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Project directory:", PROJECT_DIR)
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# Shared run settings.
MODEL_ID = "Qwen/Qwen3-4B"  # Choose: Qwen/Qwen3-1.7B, Qwen/Qwen3-4B, Qwen/Qwen3-8B, Qwen/Qwen3-14B
LAYERS = [6, 13, 20]
K = 10
SAMPLING_PROFILE = "official_recommended"
EXP_A_N_BASELINE = 30
EXP_B_N_SAMPLES = 100
EXP_C_N_SAMPLES = 200
MAX_NEW_TOKENS = 1024

MODEL_SLUG = MODEL_ID.replace("/", "--")
OUTPUT_ROOT = PROJECT_DIR / "outputs" / MODEL_SLUG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_script(args):
    print("Running:", " ".join(str(part) for part in args))
    subprocess.run([sys.executable, *map(str, args)], cwd=PROJECT_DIR, check=True)

print("Selected model:", MODEL_ID)
print("Output root:", OUTPUT_ROOT)


## 1. Download GSM8K on the Colab Machine

In [ ]:
run_script(["scripts/bootstrap_assets.py", "--skip-model"])


## 2. Download the Selected Model on the Colab Machine

In [ ]:
run_script(["scripts/bootstrap_assets.py", "--model-id", MODEL_ID, "--skip-dataset"])


## 3. Run Experiment A

In [ ]:
run_script([
    "scripts/run_exp_a.py",
    "--model-id", MODEL_ID,
    "--n-baseline", EXP_A_N_BASELINE,
    "--k", K,
    "--layers", *LAYERS,
    "--sampling-profile", SAMPLING_PROFILE,
    "--max-new-tokens", MAX_NEW_TOKENS,
    "--output-dir", OUTPUT_ROOT / "exp_a",
])


## 4. Run Experiment B (Corrected Name)

In [ ]:
# Corrected Experiment B: GSM8K prompt-structure control in no-thinking mode.
run_script([
    "scripts/run_exp_b.py",
    "--model-id", MODEL_ID,
    "--n-samples", EXP_B_N_SAMPLES,
    "--k", K,
    "--layers", *LAYERS,
    "--sampling-profile", SAMPLING_PROFILE,
    "--max-new-tokens", MAX_NEW_TOKENS,
    "--output-dir", OUTPUT_ROOT / "exp_b",
])


## 5. Run Experiment C

In [ ]:
# Corrected Experiment C: GSM8K canonical think vs no-think comparison.
run_script([
    "scripts/run_exp_c.py",
    "--model-id", MODEL_ID,
    "--n-samples", EXP_C_N_SAMPLES,
    "--k", K,
    "--layers", *LAYERS,
    "--sampling-profile", SAMPLING_PROFILE,
    "--max-new-tokens", MAX_NEW_TOKENS,
    "--output-dir", OUTPUT_ROOT / "exp_c",
])


## 6. Save All Plots and Results to Google Drive

In [ ]:
from google.colab import drive
import shutil
from datetime import datetime

drive.mount("/content/drive")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_root = Path("/content/drive/MyDrive/qwen_lid_outputs")
drive_target = drive_root / f"{MODEL_SLUG}_{timestamp}"
drive_target.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(OUTPUT_ROOT, drive_target, dirs_exist_ok=True)
archive_path = shutil.make_archive(str(drive_target), "zip", OUTPUT_ROOT)
print("Copied output folder to:", drive_target)
print("Created zip archive:", archive_path)
